In [ ]:
import sys
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
from IPython.display import display

# 设置输出编码，避免中文乱码
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8")

# 设置随机种子，保证结果可复现
GLOBAL_SEED = 20260425
np.random.seed(GLOBAL_SEED)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)


def set_cn_plot_style():
    """设置中文图表显示格式：宋体、小四号。"""
    mpl.rcParams["font.family"] = "serif"
    mpl.rcParams["font.serif"] = [
        "SimSun",
        "Songti SC",
        "STSong",
        "Noto Serif CJK SC",
        "Source Han Serif SC",
        "DejaVu Serif",
    ]
    mpl.rcParams["font.size"] = 12
    mpl.rcParams["axes.titlesize"] = 12
    mpl.rcParams["axes.labelsize"] = 12
    mpl.rcParams["xtick.labelsize"] = 12
    mpl.rcParams["ytick.labelsize"] = 12
    mpl.rcParams["legend.fontsize"] = 12
    mpl.rcParams["axes.unicode_minus"] = False


set_cn_plot_style()


def read_csv_auto(path):
    """自动尝试常见编码读取 CSV 文件。"""
    path = Path(path)
    last_error = None

    for enc in ["utf-8-sig", "utf-8", "gb18030", "gbk"]:
        try:
            return pd.read_csv(path, encoding=enc), enc
        except Exception as e:
            last_error = e

    raise RuntimeError(f"无法读取文件：{path}。最后一次报错为：{last_error}")


DATA_PATH = Path("data.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError("未找到 data.csv。请确认 notebook 与 data.csv 位于同一文件夹下。")

df, DATA_ENCODING = read_csv_auto(DATA_PATH)

# 按数据列顺序读取变量
c_city = df.columns[0]
c_code = df.columns[1]
c_year = df.columns[2]
c_treat = df.columns[3]
c_batch = df.columns[4]

c_y = df.columns[5]
c_lngdppc = df.columns[6]
c_lnpop = df.columns[7]
c_urban = df.columns[8]
c_fdi = df.columns[9]
c_retail = df.columns[10]

c_upgrade = df.columns[11]
c_m2 = df.columns[12]
c_s2 = df.columns[13]
c_s3 = df.columns[14]
c_green = df.columns[15] if len(df.columns) > 15 else None

# 变量中文名称
VAR_LABELS = {
    "y": "减污降碳压力指数",
    "treat": "低碳城市试点政策变量",
    "ln_gdp_pc": "经济发展水平",
    "ln_pop": "城市规模",
    "urban": "城市化水平",
    "fdi_gdp": "对外开放水平",
    "retail_gdp": "市场消费水平",
    "green_innovation": "绿色创新",
    "industry_upgrade_ratio": "产业结构高级化",
    "industry_rationalization_m2": "产业结构合理化",
    "share_secondary_pct": "第二产业占比",
    "share_tertiary_pct": "第三产业占比",
}


def cn(v):
    """返回变量中文名称。"""
    return VAR_LABELS.get(v, v)


# 整理基础面板数据
BASE_PANEL = pd.DataFrame({
    "city": df[c_city].astype(str),
    "city_code": df[c_code],
    "year": pd.to_numeric(df[c_year], errors="coerce"),
    "treat": pd.to_numeric(df[c_treat], errors="coerce"),
    "batch": df[c_batch],
    "y": pd.to_numeric(df[c_y], errors="coerce"),
    "ln_gdp_pc": pd.to_numeric(df[c_lngdppc], errors="coerce"),
    "ln_pop": pd.to_numeric(df[c_lnpop], errors="coerce"),
    "urban": pd.to_numeric(df[c_urban], errors="coerce"),
    "fdi_gdp": pd.to_numeric(df[c_fdi], errors="coerce"),
    "retail_gdp": pd.to_numeric(df[c_retail], errors="coerce"),
    "green_innovation": pd.to_numeric(df[c_green], errors="coerce") if c_green is not None else np.nan,
    "industry_upgrade_ratio": pd.to_numeric(df[c_upgrade], errors="coerce"),
    "industry_rationalization_m2": pd.to_numeric(df[c_m2], errors="coerce"),
    "share_secondary_pct": pd.to_numeric(df[c_s2], errors="coerce"),
    "share_tertiary_pct": pd.to_numeric(df[c_s3], errors="coerce"),
}).replace([np.inf, -np.inf], np.nan)

# 描述性统计变量列表
spec = [
    ("核心被解释变量", "y"),
    ("控制变量", "ln_gdp_pc"),
    ("控制变量", "ln_pop"),
    ("控制变量", "urban"),
    ("控制变量", "fdi_gdp"),
    ("控制变量", "retail_gdp"),
    ("中介变量", "green_innovation"),
    ("调节变量", "industry_upgrade_ratio"),
    ("调节变量", "industry_rationalization_m2"),
    ("调节变量", "share_secondary_pct"),
    ("调节变量", "share_tertiary_pct"),
]

rows = []

for category, col in spec:
    s = pd.to_numeric(BASE_PANEL[col], errors="coerce").dropna()

    rows.append({
        "变量类别": category,
        "变量名称": cn(col),
        "样本量": int(s.shape[0]),
        "均值": float(s.mean()) if len(s) else np.nan,
        "标准差": float(s.std(ddof=1)) if len(s) > 1 else np.nan,
        "最小值": float(s.min()) if len(s) else np.nan,
        "最大值": float(s.max()) if len(s) else np.nan,
    })

desc = pd.DataFrame(rows)

# 保留四位小数
for col in ["均值", "标准差", "最小值", "最大值"]:
    desc[col] = desc[col].map(lambda x: f"{x:.4f}" if pd.notna(x) else "")

print("表1 描述性统计")

display(
    desc.style
    .set_properties(**{
        "font-family": "SimSun",
        "font-size": "12pt",
        "text-align": "center",
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("font-family", "SimSun"),
                ("font-size", "12pt"),
                ("text-align", "center"),
            ],
        }
    ])
)

print(
    "样本总量 =",
    len(BASE_PANEL),
    "，城市数量 =",
    BASE_PANEL["city"].nunique(),
    "，年份范围 =",
    int(BASE_PANEL["year"].min()),
    "-",
    int(BASE_PANEL["year"].max()),
)

In [ ]:
# 基准回归结果表
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display

if "BASE_PANEL" not in globals():
    raise RuntimeError("请先运行第1段描述性统计代码。")

# 保留基准回归所需变量，并删除缺失值
d = BASE_PANEL[
    [
        "y",
        "treat",
        "city",
        "year",
        "ln_gdp_pc",
        "ln_pop",
        "urban",
        "fdi_gdp",
        "retail_gdp",
    ]
].dropna().copy()

d = d[d["treat"].isin([0, 1])]

# 逐步加入控制变量
specs = [
    [],
    ["ln_gdp_pc"],
    ["ln_gdp_pc", "urban"],
    ["ln_gdp_pc", "urban", "ln_pop"],
    ["ln_gdp_pc", "urban", "ln_pop", "fdi_gdp"],
    ["ln_gdp_pc", "urban", "ln_pop", "fdi_gdp", "retail_gdp"],
]

models = []

for controls in specs:
    rhs = 'Q("treat")'

    if controls:
        rhs += " + " + " + ".join([f'Q("{v}")' for v in controls])

    formula = f'Q("y") ~ {rhs} + C(Q("city")) + C(Q("year"))'

    model = smf.ols(formula, data=d).fit(
        cov_type="cluster",
        cov_kwds={"groups": d["city"]},
    )

    models.append(model)

BASE_DATA = d.copy()
BEST_CONTROLS = specs[-1].copy()
BEST_MODEL = models[-1]


def stars(p):
    """根据 p 值返回显著性星号。"""
    if p < 0.001:
        return "****"
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.10:
        return "*"
    return ""


def coef_se(model, var):
    """返回系数和括号内标准误。"""
    key = f'Q("{var}")'

    if key in model.params.index:
        coef = model.params[key]
        se = model.bse[key]
        p_value = model.pvalues[key]
        return f"{coef:.4f}{stars(p_value)}\n({se:.4f})"

    return ""


row_order = [
    "treat",
    "ln_gdp_pc",
    "urban",
    "ln_pop",
    "fdi_gdp",
    "retail_gdp",
    "Intercept",
]

row_name_map = {
    "Intercept": "常数项",
    "treat": cn("treat"),
    "ln_gdp_pc": cn("ln_gdp_pc"),
    "urban": cn("urban"),
    "ln_pop": cn("ln_pop"),
    "fdi_gdp": cn("fdi_gdp"),
    "retail_gdp": cn("retail_gdp"),
}

out = pd.DataFrame({
    "变量名称": [row_name_map[v] for v in row_order]
})

for i, model in enumerate(models, start=1):
    out[f"({i})"] = [coef_se(model, v) for v in row_order]

# 回归统计量
stat_rows = pd.DataFrame({
    "变量名称": [
        "R²",
        "调整后R²",
        "城市固定效应",
        "年份固定效应",
        "样本量N",
    ]
})

for i, model in enumerate(models, start=1):
    stat_rows[f"({i})"] = [
        f"{model.rsquared:.4f}",
        f"{model.rsquared_adj:.4f}",
        "是",
        "是",
        f"{int(model.nobs)}",
    ]

baseline_table = pd.concat([out, stat_rows], ignore_index=True)

print("表2 基准回归结果（逐步加入控制变量）")
print("注：括号内为按城市聚类的稳健标准误；****、***、**、* 分别表示在0.1%、1%、5%和10%的显著性水平上显著。")

display(
    baseline_table.style
    .set_properties(**{
        "font-family": "SimSun",
        "font-size": "12pt",
        "text-align": "center",
        "white-space": "pre-wrap",
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("font-family", "SimSun"),
                ("font-size", "12pt"),
                ("text-align", "center"),
            ],
        }
    ])
)

In [ ]:
# 平行趋势检验
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

if "set_cn_plot_style" in globals():
    set_cn_plot_style()

if "BASE_DATA" not in globals() or "BEST_CONTROLS" not in globals():
    raise RuntimeError("请先运行第2段基准回归代码。")

# 关闭之前残留的所有图，避免重复输出
plt.close("all")

d = BASE_DATA.copy()
controls = BEST_CONTROLS.copy()

WINDOW = (-3, 4)
BASE_PERIOD = -1
lo, hi = WINDOW

first_treat_year = d[d["treat"] == 1].groupby("city")["year"].min()

d["g"] = d["city"].map(first_treat_year)
d["ever"] = d["g"].notna().astype(int)
d["rel"] = d["year"] - d["g"]
d["rel_cap"] = d["rel"]

d.loc[d["ever"] == 1, "rel_cap"] = d.loc[d["ever"] == 1, "rel_cap"].clip(lo, hi)

event_periods = list(range(lo, hi + 1))

if BASE_PERIOD in event_periods:
    event_periods.remove(BASE_PERIOD)


def event_var_name(k):
    return f"政策前{abs(k)}期" if k < 0 else f"政策后{k}期"


term_map = {}

for k in event_periods:
    var_name = event_var_name(k)
    term_map[k] = var_name
    d[var_name] = ((d["ever"] == 1) & (d["rel_cap"] == k)).astype(int)

rhs = " + ".join(
    [f'Q("{term_map[k]}")' for k in event_periods]
    + [f'Q("{v}")' for v in controls]
)

formula = f'Q("y") ~ {rhs} + C(Q("city")) + C(Q("year"))'

event_model = smf.ols(formula, data=d).fit(
    cov_type="cluster",
    cov_kwds={"groups": d["city"]},
)

lead_terms = [
    f'Q("{term_map[k]}")'
    for k in range(lo, 0)
    if k != BASE_PERIOD and k in term_map
]

if len(lead_terms) > 0:
    hypothesis = " = 0, ".join(lead_terms) + " = 0"
    wald_result = event_model.wald_test(hypothesis)
    print(f"政策前趋势联合检验 p 值（事件窗口 {WINDOW}）= {float(wald_result.pvalue):.6f}")

plot_periods = list(range(lo, hi + 1))
coefs = []
ci_low = []
ci_high = []

for k in plot_periods:
    if k == BASE_PERIOD:
        coefs.append(0.0)
        ci_low.append(0.0)
        ci_high.append(0.0)
    else:
        term = f'Q("{term_map[k]}")'
        coef = event_model.params.get(term, np.nan)
        se = event_model.bse.get(term, np.nan)

        coefs.append(coef)
        ci_low.append(coef - 1.96 * se)
        ci_high.append(coef + 1.96 * se)

# 显示事件研究图
fig, ax = plt.subplots(figsize=(8.2, 5.0), dpi=600)

ax.errorbar(
    plot_periods,
    coefs,
    yerr=[
        np.array(coefs) - np.array(ci_low),
        np.array(ci_high) - np.array(coefs),
    ],
    fmt="o-",
    color="#333333",
    ecolor="#666666",
    elinewidth=1.2,
    capsize=3,
    markersize=4,
)

ax.axhline(0, color="gray", linestyle="--", linewidth=1)
ax.axvline(0, color="gray", linestyle="--", linewidth=1)

ax.set_xticks(plot_periods)
ax.set_xlabel("试点实施前后相对时间")
ax.set_ylabel("估计系数")
ax.set_title("平行趋势检验：事件研究图")
ax.grid(False)

plt.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
# Bootstrap 安慰剂检验
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib

try:
    from IPython import get_ipython

    ipython = get_ipython()
    if ipython is not None and "IPKernelApp" in ipython.config:
        ipython.run_line_magic("matplotlib", "inline")
        ipython.run_line_magic("config", "InlineBackend.figure_format = 'retina'")
    else:
        matplotlib.use("Agg")
except Exception:
    matplotlib.use("Agg")

import matplotlib.pyplot as plt

if "set_cn_plot_style" in globals():
    set_cn_plot_style()

if "BASE_DATA" not in globals() or "BEST_CONTROLS" not in globals() or "BEST_MODEL" not in globals():
    raise RuntimeError("请先运行第2段基准回归代码。")

# 关闭之前残留图像，避免重复输出
plt.close("all")

d = BASE_DATA.copy()
controls = BEST_CONTROLS.copy()

real_beta = float(BEST_MODEL.params['Q("treat")'])
real_p = float(BEST_MODEL.pvalues['Q("treat")'])

print(f"真实政策系数 = {real_beta:.6f}，P值 = {real_p:.6f}")

B = 300
rng = np.random.default_rng(20260418)

# 获取真实试点城市数量与真实政策开始年份分布
real_start_year = d[d["treat"] == 1].groupby("city")["year"].min()
all_cities = np.array(sorted(d["city"].unique()))
num_treated = len(real_start_year.index)
start_pool = real_start_year.values

betas = []
pvals = []

formula_fake = (
    'Q("y") ~ fake_treat + '
    + " + ".join([f'Q("{v}")' for v in controls])
    + ' + C(Q("city")) + C(Q("year"))'
)

# 随机抽取虚假试点城市和虚假政策开始年份
for _ in range(B):
    fake_cities = rng.choice(all_cities, size=num_treated, replace=False)
    fake_starts = rng.choice(start_pool, size=num_treated, replace=True)
    start_map = pd.Series(fake_starts, index=fake_cities)

    dd = d.copy()
    start_vec = dd["city"].map(start_map)
    dd["fake_treat"] = ((dd["year"] >= start_vec) & start_vec.notna()).astype(int)

    try:
        model = smf.ols(formula_fake, data=dd).fit(
            cov_type="cluster",
            cov_kwds={"groups": dd["city"]},
        )

        betas.append(float(model.params["fake_treat"]))
        pvals.append(float(model.pvalues["fake_treat"]))

    except Exception:
        continue

betas = np.array(betas)
pvals = np.array(pvals)

print(f"有效安慰剂回归次数：{len(betas)} / {B}")

if len(betas) == 0:
    raise RuntimeError("有效安慰剂回归次数为0，请检查数据或模型设定。")

emp_p = np.mean(np.abs(betas) >= abs(real_beta))
print(f"经验安慰剂P值 = {emp_p:.6f}")

# 绘制安慰剂检验图
fig, ax1 = plt.subplots(figsize=(7.6, 5.0), dpi=600)

try:
    from scipy.stats import gaussian_kde

    xs = np.linspace(betas.min() - 0.05, betas.max() + 0.05, 400)
    ys = gaussian_kde(betas)(xs)

    ax1.plot(xs, ys, color="black", lw=1.2, label="核密度")

except Exception:
    ax1.hist(
        betas,
        bins=30,
        density=True,
        color="lightgray",
        edgecolor="gray",
        label="系数分布",
    )

ax1.axvline(real_beta, color="gray", linestyle=":", linewidth=1.2, label="真实政策系数")
ax1.set_xlabel("回归系数")
ax1.set_ylabel("核密度")
ax1.set_title("安慰剂检验图")

ax2 = ax1.twinx()
ax2.scatter(
    betas,
    pvals,
    s=14,
    facecolors="none",
    edgecolors="#9a9a9a",
    alpha=0.85,
    label="P值",
)

ax2.axhline(0.10, color="gray", linestyle="--", linewidth=1.0, label="10%显著性水平")
ax2.set_ylabel("P值")
ax2.set_ylim(-0.02, 1.02)

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    handles1 + handles2,
    labels1 + labels2,
    frameon=False,
    loc="upper right",
)

plt.tight_layout()
plt.show()

# 显示完成后关闭，防止后续单元格重复显示
plt.close(fig)

In [ ]:
# 排除异常值干扰的稳健性检验：缩尾处理
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display

if "BASE_DATA" not in globals() or "BEST_CONTROLS" not in globals():
    raise RuntimeError("请先运行第2段基准回归代码。")

# 设置双侧缩尾比例
WINSOR_P = 0.01

d0 = BASE_DATA.copy()
controls = BEST_CONTROLS.copy()

# 构造固定效应变量
if "city_fe" not in d0.columns:
    d0["city_fe"] = d0["city"].astype(str)

if "year_fe" not in d0.columns:
    d0["year_fe"] = d0["year"].astype(int).astype(str)

winsor_cols = ["y"] + controls


def winsorize_series(s, p=0.01):
    """对变量进行双侧缩尾处理。"""
    lo = s.quantile(p)
    hi = s.quantile(1 - p)
    return s.clip(lower=lo, upper=hi), lo, hi


# 对被解释变量和控制变量进行缩尾
d1 = d0.copy()
bounds = []

for col in winsor_cols:
    x = pd.to_numeric(d1[col], errors="coerce")
    xw, lo, hi = winsorize_series(x, p=WINSOR_P)
    d1[col] = xw
    bounds.append((col, float(lo), float(hi)))

# 与基准回归保持一致的 DID 设定
formula = (
    'Q("y") ~ Q("treat") + '
    + " + ".join([f'Q("{v}")' for v in controls])
    + ' + C(Q("city_fe")) + C(Q("year_fe"))'
)

model_before = smf.ols(formula, data=d0).fit(
    cov_type="cluster",
    cov_kwds={"groups": d0["city"]},
)

model_after = smf.ols(formula, data=d1).fit(
    cov_type="cluster",
    cov_kwds={"groups": d1["city"]},
)


def stars(p):
    """根据 p 值返回显著性星号。"""
    if p < 0.001:
        return "****"
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.10:
        return "*"
    return ""


def collect_result(model, name):
    """提取核心变量回归结果。"""
    b = float(model.params['Q("treat")'])
    se = float(model.bse['Q("treat")'])
    p = float(model.pvalues['Q("treat")'])
    ci = model.conf_int().loc['Q("treat")'].tolist()

    return {
        "模型": name,
        "treat系数": f"{b:.6f}{stars(p)}",
        "聚类稳健标准误": f"{se:.6f}",
        "P值": f"{p:.6f}",
        "95%置信区间": f"[{float(ci[0]):.6f}, {float(ci[1]):.6f}]",
        "样本量": int(model.nobs),
        "R²": f"{float(model.rsquared):.6f}",
    }


robust_table = pd.DataFrame([
    collect_result(model_before, "缩尾前"),
    collect_result(model_after, "缩尾后"),
])

bounds_table = pd.DataFrame(
    bounds,
    columns=["变量名称", "缩尾下界", "缩尾上界"],
)

bounds_table["变量名称"] = bounds_table["变量名称"].map(lambda x: cn(x) if "cn" in globals() else x)
bounds_table["缩尾下界"] = bounds_table["缩尾下界"].map(lambda x: f"{x:.6f}")
bounds_table["缩尾上界"] = bounds_table["缩尾上界"].map(lambda x: f"{x:.6f}")

print("表5 排除异常值干扰的稳健性检验")
print(f"缩尾方式：对被解释变量和控制变量进行双侧 {WINSOR_P * 100:.1f}% 缩尾处理")
print("模型设定：减污降碳压力指数 ~ 低碳城市试点政策变量 + 控制变量 + 城市固定效应 + 年份固定效应")
print("注：标准误为按城市聚类的稳健标准误；****、***、**、* 分别表示在0.1%、1%、5%和10%的显著性水平上显著。")

display(
    robust_table.style
    .set_properties(**{
        "font-family": "SimSun",
        "font-size": "12pt",
        "text-align": "center",
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("font-family", "SimSun"),
                ("font-size", "12pt"),
                ("text-align", "center"),
            ],
        }
    ])
)

print("缩尾边界：")

display(
    bounds_table.style
    .set_properties(**{
        "font-family": "SimSun",
        "font-size": "12pt",
        "text-align": "center",
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("font-family", "SimSun"),
                ("font-size", "12pt"),
                ("text-align", "center"),
            ],
        }
    ])
)

In [ ]:
# PSM-DID稳健性检验：基于政策前城市特征的倾向得分匹配
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display

if "BASE_PANEL" not in globals() or "BEST_CONTROLS" not in globals() or "BEST_MODEL" not in globals():
    raise RuntimeError("请先运行第1段和第2段代码。")

K_MATCH = 3
CALIPER = 0.05
SEED = 20260418

rng = np.random.default_rng(SEED)
controls = BEST_CONTROLS.copy()

# 构造用于匹配和回归的面板数据
d = BASE_PANEL[["city", "year", "treat", "y"] + controls].dropna().copy()
d = d[d["treat"].isin([0, 1])]

# 识别城市首次进入处理状态的年份
first_treat = d[d["treat"] == 1].groupby("city")["year"].min()

city_df = pd.DataFrame({
    "city": sorted(d["city"].unique())
})

city_df["g"] = city_df["city"].map(first_treat)
city_df["ever_treated"] = city_df["g"].notna().astype(int)

if city_df["ever_treated"].sum() == 0:
    raise RuntimeError("未识别到处理组城市。")

# 使用最早试点年份之前的样本构造政策前特征
global_g0 = int(city_df.loc[city_df["ever_treated"] == 1, "g"].min())
pre = d[d["year"] < global_g0].copy()

if pre.empty:
    raise RuntimeError("未找到政策前观测值，请检查政策开始年份。")


def slope_by_city(tmp, ycol):
    """计算各城市政策前控制变量的时间趋势。"""
    out = {}

    for city, group in tmp.groupby("city"):
        gx = group[["year", ycol]].dropna().copy()

        if len(gx) < 2:
            out[city] = np.nan
            continue

        x = gx["year"].to_numpy(dtype=float)
        y = gx[ycol].to_numpy(dtype=float)

        x = x - x.mean()
        denom = np.sum(x ** 2)

        if denom <= 0:
            out[city] = np.nan
        else:
            out[city] = float(np.sum(x * (y - y.mean())) / denom)

    return pd.Series(out)


# 构造城市层面的政策前均值和趋势特征
feat = pd.DataFrame({
    "city": sorted(pre["city"].unique())
})

for v in controls:
    mean_feature = pre.groupby("city")[v].mean().rename(f"{v}_mean")
    trend_feature = slope_by_city(pre, v).rename(f"{v}_trend")

    feat = feat.merge(mean_feature, left_on="city", right_index=True, how="left")
    feat = feat.merge(trend_feature, left_on="city", right_index=True, how="left")

city_match = city_df.merge(feat, on="city", how="left")

feature_cols = [
    col for col in city_match.columns
    if col.endswith("_mean") or col.endswith("_trend")
]

city_match = city_match.dropna(subset=feature_cols).copy()

treated_cities = city_match.loc[city_match["ever_treated"] == 1, "city"].tolist()
control_cities = city_match.loc[city_match["ever_treated"] == 0, "city"].tolist()

if len(treated_cities) == 0 or len(control_cities) == 0:
    raise RuntimeError("PSM需要同时存在处理组城市和从未处理组城市。")

# 估计倾向得分
X = city_match[feature_cols].astype(float).copy()
X = (X - X.mean()) / X.std(ddof=0).replace(0, np.nan)
X = X.fillna(0.0)

y_psm = city_match["ever_treated"].astype(int).values

use_sklearn = True

try:
    from sklearn.linear_model import LogisticRegression
except Exception:
    use_sklearn = False

if use_sklearn:
    lr = LogisticRegression(
        max_iter=3000,
        solver="lbfgs",
        random_state=SEED,
    )
    lr.fit(X.values, y_psm)
    city_match["pscore"] = lr.predict_proba(X.values)[:, 1]
else:
    import statsmodels.api as sm

    X2 = sm.add_constant(X, has_constant="add")
    glm = sm.GLM(y_psm, X2, family=sm.families.Binomial()).fit()
    city_match["pscore"] = glm.predict(X2)

# 共同支撑区域筛选
ps_t = city_match.loc[city_match["ever_treated"] == 1, "pscore"]
ps_c = city_match.loc[city_match["ever_treated"] == 0, "pscore"]

lo = max(float(ps_t.min()), float(ps_c.min()))
hi = min(float(ps_t.max()), float(ps_c.max()))

cm = city_match[
    (city_match["pscore"] >= lo)
    & (city_match["pscore"] <= hi)
].copy()

treated_match = cm[cm["ever_treated"] == 1].copy()
control_match = cm[cm["ever_treated"] == 0].copy()

if treated_match.empty or control_match.empty:
    raise RuntimeError("共同支撑区域筛选后没有足够样本。")

# K近邻匹配，允许重复匹配
control_ps = control_match["pscore"].to_numpy()
control_city = control_match["city"].to_numpy()

matched_pairs = []

for _, row in treated_match.iterrows():
    diff = np.abs(control_ps - float(row["pscore"]))
    order = np.argsort(diff)

    in_caliper = order[diff[order] <= CALIPER]

    if len(in_caliper) >= K_MATCH:
        pick = in_caliper[:K_MATCH]
    elif len(in_caliper) > 0:
        need = K_MATCH - len(in_caliper)
        outside = [j for j in order if j not in set(in_caliper)]
        pick = np.concatenate([in_caliper, np.array(outside[:need], dtype=int)])
    else:
        pick = order[:K_MATCH]

    for j in pick:
        matched_pairs.append((row["city"], control_city[j]))

pairs = pd.DataFrame(
    matched_pairs,
    columns=["处理组城市", "匹配控制组城市"],
)

# 根据匹配频次构造权重
ctrl_w = pairs["匹配控制组城市"].value_counts().rename("match_count").to_frame()

treated_set = set(pairs["处理组城市"].unique())
control_set = set(pairs["匹配控制组城市"].unique())

matched_cities = sorted(treated_set.union(control_set))
panel_m = d[d["city"].isin(matched_cities)].copy()

panel_m["w"] = 0.0
panel_m.loc[panel_m["city"].isin(treated_set), "w"] = 1.0

panel_m = panel_m.merge(
    ctrl_w,
    left_on="city",
    right_index=True,
    how="left",
)

panel_m.loc[panel_m["city"].isin(control_set), "w"] = (
    panel_m.loc[panel_m["city"].isin(control_set), "match_count"] / float(K_MATCH)
)

panel_m["w"] = panel_m["w"].fillna(0.0)
panel_m = panel_m[panel_m["w"] > 0].copy()

panel_m["city_fe"] = panel_m["city"].astype(str)
panel_m["year_fe"] = panel_m["year"].astype(int).astype(str)

# 在匹配后的样本上进行加权DID回归
formula = (
    'Q("y") ~ Q("treat") + '
    + " + ".join([f'Q("{v}")' for v in controls])
    + ' + C(Q("city_fe")) + C(Q("year_fe"))'
)

m_psm = smf.wls(
    formula,
    data=panel_m,
    weights=panel_m["w"],
).fit(
    cov_type="cluster",
    cov_kwds={"groups": panel_m["city"]},
)


def stars(p):
    """根据p值返回显著性星号。"""
    if p < 0.001:
        return "****"
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.10:
        return "*"
    return ""


def get_treat_result(model, model_name):
    """提取政策变量回归结果。"""
    key = 'Q("treat")' if 'Q("treat")' in model.params.index else "treat"

    b = float(model.params[key])
    se = float(model.bse[key])
    p = float(model.pvalues[key])
    ci = model.conf_int().loc[key].tolist()

    return {
        "模型": model_name,
        "treat系数": f"{b:.6f}{stars(p)}",
        "聚类稳健标准误": f"{se:.6f}",
        "P值": f"{p:.6f}",
        "95%置信区间": f"[{float(ci[0]):.6f}, {float(ci[1]):.6f}]",
        "样本量": int(model.nobs),
        "R²": f"{float(model.rsquared):.6f}",
    }


psm_result_table = pd.DataFrame([
    get_treat_result(BEST_MODEL, "基准DID（全样本）"),
    get_treat_result(m_psm, "PSM-DID（匹配样本）"),
])

print("表6 PSM-DID稳健性检验")
print(f"政策前样本年份：{global_g0}年以前")
print(f"匹配方式：1:{K_MATCH} 近邻匹配，卡尺值 = {CALIPER}，允许重复匹配")
print(f"共同支撑区域城市数：{len(cm)}，其中处理组城市数：{int((cm['ever_treated'] == 1).sum())}，控制组城市数：{int((cm['ever_treated'] == 0).sum())}")
print(f"匹配后处理组城市数：{len(treated_set)}，匹配后控制组城市数：{len(control_set)}")
print("注：标准误为按城市聚类的稳健标准误；****、***、**、* 分别表示在0.1%、1%、5%和10%的显著性水平上显著。")

display(
    psm_result_table.style
    .set_properties(**{
        "font-family": "SimSun",
        "font-size": "12pt",
        "text-align": "center",
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("font-family", "SimSun"),
                ("font-size", "12pt"),
                ("text-align", "center"),
            ],
        }
    ])
)

# 匹配平衡性检验：标准化均值差
def weighted_mean(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    s = np.sum(w)
    return np.nan if s <= 0 else float(np.sum(w * x) / s)


def weighted_var(x, w):
    mu = weighted_mean(x, w)

    if np.isnan(mu):
        return np.nan

    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    s = np.sum(w)

    return np.nan if s <= 0 else float(np.sum(w * (x - mu) ** 2) / s)


def smd_weighted(x_t, w_t, x_c, w_c):
    mt = weighted_mean(x_t, w_t)
    mc = weighted_mean(x_c, w_c)
    vt = weighted_var(x_t, w_t)
    vc = weighted_var(x_c, w_c)

    sp = np.sqrt((vt + vc) / 2.0)

    if np.isnan(sp) or sp == 0:
        return np.nan

    return float((mt - mc) / sp)


cm_before = cm.copy()

city_w = pd.DataFrame({
    "city": sorted(cm["city"].unique())
})

city_w["w_t_after"] = city_w["city"].isin(treated_set).astype(float)

city_w = city_w.merge(
    ctrl_w,
    left_on="city",
    right_index=True,
    how="left",
)

city_w["w_c_after"] = city_w["match_count"].fillna(0.0) / float(K_MATCH)
city_w = city_w.drop(columns=["match_count"])

cm_after = cm.merge(city_w, on="city", how="left")

balance_rows = []

for v in controls:
    fv = f"{v}_mean"

    xt0 = cm_before.loc[cm_before["ever_treated"] == 1, fv].to_numpy()
    wt0 = np.ones(len(xt0))
    xc0 = cm_before.loc[cm_before["ever_treated"] == 0, fv].to_numpy()
    wc0 = np.ones(len(xc0))

    smd_before = smd_weighted(xt0, wt0, xc0, wc0)

    xt1 = cm_after.loc[cm_after["ever_treated"] == 1, fv].to_numpy()
    wt1 = cm_after.loc[cm_after["ever_treated"] == 1, "w_t_after"].fillna(0.0).to_numpy()
    xc1 = cm_after.loc[cm_after["ever_treated"] == 0, fv].to_numpy()
    wc1 = cm_after.loc[cm_after["ever_treated"] == 0, "w_c_after"].fillna(0.0).to_numpy()

    smd_after = smd_weighted(xt1, wt1, xc1, wc1)

    balance_rows.append({
        "变量名称": cn(v) if "cn" in globals() else v,
        "匹配前SMD": f"{smd_before:.4f}" if pd.notna(smd_before) else "",
        "匹配后SMD": f"{smd_after:.4f}" if pd.notna(smd_after) else "",
    })

balance_table = pd.DataFrame(balance_rows)

print("匹配平衡性检验：标准化均值差（SMD越接近0，说明匹配效果越好）")

display(
    balance_table.style
    .set_properties(**{
        "font-family": "SimSun",
        "font-size": "12pt",
        "text-align": "center",
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("font-family", "SimSun"),
                ("font-size", "12pt"),
                ("text-align", "center"),
            ],
        }
    ])
)

In [ ]:
# 加入控制变量与时间趋势交互项的稳健性检验
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display

if "BASE_DATA" not in globals() or "BEST_CONTROLS" not in globals():
    raise RuntimeError("请先运行第2段基准回归代码。")

d = BASE_DATA.copy()
controls = BEST_CONTROLS.copy()

# 构造固定效应变量
d["city_fe"] = d["city"].astype(str)
d["year_fe"] = d["year"].astype(int).astype(str)

# 构造线性时间趋势变量，起点设为0
d["trend"] = d["year"] - d["year"].min()

# 基准DID模型
formula_base = (
    'Q("y") ~ Q("treat") + '
    + " + ".join([f'Q("{v}")' for v in controls])
    + ' + C(Q("city_fe")) + C(Q("year_fe"))'
)

model_base = smf.ols(
    formula_base,
    data=d,
).fit(
    cov_type="cluster",
    cov_kwds={"groups": d["city"]},
)

# 加入“控制变量 × 时间趋势”交互项
interaction_terms = " + ".join([f'Q("{v}"):Q("trend")' for v in controls])

formula_trend = (
    'Q("y") ~ Q("treat") + '
    + " + ".join([f'Q("{v}")' for v in controls])
    + " + "
    + interaction_terms
    + ' + C(Q("city_fe")) + C(Q("year_fe"))'
)

model_trend = smf.ols(
    formula_trend,
    data=d,
).fit(
    cov_type="cluster",
    cov_kwds={"groups": d["city"]},
)


def stars(p):
    """根据p值返回显著性星号。"""
    if p < 0.001:
        return "****"
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.10:
        return "*"
    return ""


def get_param_key(model, var):
    """兼容Q()变量名与普通变量名。"""
    q_key = f'Q("{var}")'
    if q_key in model.params.index:
        return q_key
    return var


def collect_result(model, model_name):
    """提取政策变量回归结果。"""
    key = get_param_key(model, "treat")

    coef = float(model.params[key])
    se = float(model.bse[key])
    p_value = float(model.pvalues[key])
    ci = model.conf_int().loc[key].tolist()

    return {
        "模型": model_name,
        "treat系数": f"{coef:.6f}{stars(p_value)}",
        "聚类稳健标准误": f"{se:.6f}",
        "P值": f"{p_value:.6f}",
        "95%置信区间": f"[{float(ci[0]):.6f}, {float(ci[1]):.6f}]",
        "样本量": int(model.nobs),
        "R²": f"{float(model.rsquared):.6f}",
    }


trend_table = pd.DataFrame([
    collect_result(model_base, "基准DID"),
    collect_result(model_trend, "加入控制变量×时间趋势"),
])

print("表X 加入控制变量与时间趋势交互项的稳健性检验")
print("模型设定：在基准DID模型基础上加入控制变量与线性时间趋势的交互项。")
print("注：标准误为按城市聚类的稳健标准误；****、***、**、* 分别表示在0.1%、1%、5%和10%的显著性水平上显著。")

display(
    trend_table.style
    .set_properties(**{
        "font-family": "SimSun",
        "font-size": "12pt",
        "text-align": "center",
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("font-family", "SimSun"),
                ("font-size", "12pt"),
                ("text-align", "center"),
            ],
        }
    ])
)

In [ ]:
# Tobit稳健性检验
import numpy as np
import pandas as pd
import patsy
from scipy.stats import norm
from statsmodels.base.model import GenericLikelihoodModel
from IPython.display import display

if "BASE_PANEL" not in globals() or "BEST_CONTROLS" not in globals():
    raise RuntimeError("请先运行第1段和第2段代码。")

controls = BEST_CONTROLS.copy()

# 保持与基准回归一致的样本和变量
need = ["y", "treat", "city", "year"] + controls

d = (
    BASE_PANEL[need]
    .copy()
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

d = d[d["treat"].isin([0, 1])]

# Tobit上下界设定：减污降碳压力指数理论上位于[0,1]
LEFT = 0.0
RIGHT = 1.0
EPS = 1e-10

formula = (
    "y ~ treat + "
    + " + ".join(controls)
    + " + C(city) + C(year)"
)

y_df, X_df = patsy.dmatrices(
    formula,
    d,
    return_type="dataframe",
)

y = y_df.iloc[:, 0].to_numpy(dtype=float)
X = X_df.to_numpy(dtype=float)


class Tobit(GenericLikelihoodModel):
    """双侧删失Tobit模型。"""

    def __init__(self, endog, exog, left=0.0, right=1.0, **kwargs):
        self.left = left
        self.right = right
        super().__init__(endog, exog, **kwargs)

    def nloglikeobs(self, params):
        beta = params[:-1]
        sigma = np.exp(params[-1])
        mu = self.exog @ beta
        yv = self.endog

        z_left = (self.left - mu) / sigma
        z_right = (self.right - mu) / sigma
        z_mid = (yv - mu) / sigma

        ll = np.empty_like(yv, dtype=float)

        left_mask = yv <= self.left + EPS
        right_mask = yv >= self.right - EPS
        mid_mask = ~(left_mask | right_mask)

        ll[left_mask] = norm.logcdf(z_left[left_mask])
        ll[right_mask] = norm.logsf(z_right[right_mask])
        ll[mid_mask] = norm.logpdf(z_mid[mid_mask]) - np.log(sigma)

        return -ll


# 使用OLS结果作为Tobit估计初始值
beta0, *_ = np.linalg.lstsq(X, y, rcond=None)
resid = y - X @ beta0
sigma0 = np.std(resid) if np.std(resid) > 1e-8 else 1.0
start = np.r_[beta0, np.log(sigma0)]

tobit_model = Tobit(y, X, left=LEFT, right=RIGHT)

tobit_result = tobit_model.fit(
    start_params=start,
    method="bfgs",
    maxiter=300,
    disp=False,
)

names = list(X_df.columns) + ["ln_sigma"]

params = pd.Series(tobit_result.params, index=names)
bse = pd.Series(tobit_result.bse, index=names)
pval = pd.Series(tobit_result.pvalues, index=names)


def stars(p):
    """根据p值返回显著性星号。"""
    if p < 0.001:
        return "****"
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.10:
        return "*"
    return ""


def find_treat_name(index):
    """查找政策变量名称。"""
    if "treat" in index:
        return "treat"

    candidates = [name for name in index if name.strip() == "treat"]

    if candidates:
        return candidates[0]

    raise RuntimeError("未在Tobit估计结果中找到treat变量。")


treat_name = find_treat_name(params.index)

tobit_coef = float(params[treat_name])
tobit_se = float(bse[treat_name])
tobit_p = float(pval[treat_name])

converged = bool(
    getattr(tobit_result, "mle_retvals", {}).get("converged", True)
)

# 整理Tobit结果
rows = [
    {
        "模型": "Tobit模型",
        "treat系数": f"{tobit_coef:.6f}{stars(tobit_p)}",
        "标准误": f"{tobit_se:.6f}",
        "P值": f"{tobit_p:.6f}",
        "样本量": len(y),
        "城市数量": d["city"].nunique(),
        "是否收敛": "是" if converged else "否",
    }
]

# 与基准DID模型进行对比
if "BEST_MODEL" in globals():
    if 'Q("treat")' in BEST_MODEL.params.index:
        ols_key = 'Q("treat")'
    elif "treat" in BEST_MODEL.params.index:
        ols_key = "treat"
    else:
        ols_key = None

    if ols_key is not None:
        ols_coef = float(BEST_MODEL.params[ols_key])
        ols_se = float(BEST_MODEL.bse[ols_key])
        ols_p = float(BEST_MODEL.pvalues[ols_key])

        rows.append({
            "模型": "基准DID模型",
            "treat系数": f"{ols_coef:.6f}{stars(ols_p)}",
            "标准误": f"{ols_se:.6f}",
            "P值": f"{ols_p:.6f}",
            "样本量": int(BEST_MODEL.nobs),
            "城市数量": d["city"].nunique(),
            "是否收敛": "不适用",
        })

tobit_table = pd.DataFrame(rows)

print("表X Tobit稳健性检验")
print("模型设定：考虑到被解释变量为[0,1]区间内的受限指标，采用双侧删失Tobit模型进行补充稳健性检验。")
print("注：Tobit检验为补充稳健性检验，本文的基准识别仍以双向固定效应DID模型为主。")
print("注：****、***、**、* 分别表示在0.1%、1%、5%和10%的显著性水平上显著。")

display(
    tobit_table.style
    .set_properties(**{
        "font-family": "SimSun",
        "font-size": "12pt",
        "text-align": "center",
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("font-family", "SimSun"),
                ("font-size", "12pt"),
                ("text-align", "center"),
            ],
        }
    ])
)

In [ ]:
# 稳健性检验汇总表
import pandas as pd
import numpy as np
from IPython.display import display

def sig_stars(p):
    """根据p值返回显著性星号。"""
    if p < 0.001:
        return "****"
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.10:
        return "*"
    return ""


def get_key(model, var="treat"):
    """兼容 Q('treat') 和 treat 两种变量名。"""
    q_key = f'Q("{var}")'
    if q_key in model.params.index:
        return q_key
    if var in model.params.index:
        return var
    raise KeyError(f"未找到变量：{var}")


def get_result_from_model(model):
    """从回归模型中提取treat系数、标准误、p值、样本量和R²。"""
    key = get_key(model, "treat")

    coef = float(model.params[key])
    se = float(model.bse[key])
    p = float(model.pvalues[key])

    coef_text = f"{coef:.4f}{sig_stars(p)}"
    se_text = f"({se:.4f})"

    n = int(model.nobs) if hasattr(model, "nobs") else ""
    r2 = f"{float(model.rsquared):.4f}" if hasattr(model, "rsquared") else "—"

    return coef_text, se_text, n, r2


def get_result_from_tobit():
    """提取Tobit模型中的treat结果。"""
    if "params" not in globals() or "bse" not in globals() or "pval" not in globals():
        return "—", "—", "—", "—"

    if "treat" in params.index:
        key = "treat"
    else:
        candidates = [x for x in params.index if str(x).strip() == "treat"]
        if len(candidates) == 0:
            return "—", "—", "—", "—"
        key = candidates[0]

    coef = float(params[key])
    se = float(bse[key])
    p = float(pval[key])

    coef_text = f"{coef:.4f}{sig_stars(p)}"
    se_text = f"({se:.4f})"

    if "y" in globals():
        n = len(y)
    else:
        n = "—"

    r2 = "—"

    return coef_text, se_text, n, r2


# 提取各稳健性检验模型
robust_models = {}

# 排除异常值干扰：缩尾后模型
if "model_after" in globals():
    robust_models["排除异常值"] = model_after
elif "m_after" in globals():
    robust_models["排除异常值"] = m_after

# 时间趋势：加入控制变量×时间趋势后的模型
if "model_trend" in globals():
    robust_models["时间趋势"] = model_trend
elif "m1" in globals():
    robust_models["时间趋势"] = m1

# PSM-DID：匹配后模型
if "m_psm" in globals():
    robust_models["PSM-DID"] = m_psm

# Tobit：单独处理
has_tobit = "params" in globals() and "bse" in globals() and "pval" in globals()

# 生成表格列
columns = []
data = {
    "变量": [
        "did",
        "",
        "控制变量",
        "城市固定效应",
        "年份固定效应",
        "N",
        "R²",
    ]
}

for model_name, model in robust_models.items():
    coef_text, se_text, n, r2 = get_result_from_model(model)

    data[model_name] = [
        coef_text,
        se_text,
        "是",
        "是",
        "是",
        n,
        r2,
    ]

if has_tobit:
    coef_text, se_text, n, r2 = get_result_from_tobit()

    data["Tobit"] = [
        coef_text,
        se_text,
        "是",
        "是",
        "是",
        n,
        r2,
    ]

robust_summary_table = pd.DataFrame(data)

print("表X 稳健性检验")
print("Table X Robustness test")
print("注：括号内为标准误；****、***、**、* 分别表示在0.1%、1%、5%和10%的显著性水平上显著。")
print("注：Tobit为补充稳健性检验，R²不适用。")

display(
    robust_summary_table.style
    .set_properties(**{
        "font-family": "SimSun",
        "font-size": "12pt",
        "text-align": "center",
        "white-space": "pre-wrap",
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("font-family", "SimSun"),
                ("font-size", "12pt"),
                ("text-align", "center"),
            ],
        }
    ])
)

In [ ]:
# 机制分析：中介效应检验
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display

if "BASE_PANEL" not in globals() or "BEST_CONTROLS" not in globals():
    raise RuntimeError("请先运行第1段和第2段代码。")

controls = BEST_CONTROLS.copy()

MEDIATOR = "green_innovation"
MEDIATOR_LABEL = "绿色创新"
Y_LABEL = "减污降碳压力指数"

BASE_B = 500
SEED = 20260420


def stars(p):
    """根据p值返回显著性星号。"""
    if p < 0.001:
        return "****"
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.10:
        return "*"
    return ""


def fmt_coef(model, var):
    """输出系数和标准误。"""
    if var not in model.params.index:
        return ""

    coef = float(model.params[var])
    se = float(model.bse[var])
    p = float(model.pvalues[var])

    return f"{coef:.4f}{stars(p)}\n({se:.4f})"


def bootstrap_indirect_cluster(df_in, mediator_var, controls_list, B=500, seed=20260420):
    """按城市聚类Bootstrap估计间接效应。"""
    rng = np.random.default_rng(seed)
    cities = np.array(sorted(df_in["city"].unique()))
    n_city = len(cities)

    rhs_c = " + ".join(controls_list) if controls_list else "1"

    formula_a = f"{mediator_var} ~ treat + {rhs_c} + C(city_bs) + C(year)"
    formula_b = f"y ~ treat + {mediator_var} + {rhs_c} + C(city_bs) + C(year)"

    values = []

    for _ in range(B):
        draw = rng.choice(cities, size=n_city, replace=True)

        parts = []

        for j, city in enumerate(draw):
            sub = df_in[df_in["city"] == city].copy()
            sub["city_bs"] = f"{city}__{j}"
            parts.append(sub)

        bs = pd.concat(parts, ignore_index=True)

        try:
            model_a = smf.ols(formula_a, data=bs).fit()
            model_b = smf.ols(formula_b, data=bs).fit()

            a = model_a.params.get("treat", np.nan)
            b = model_b.params.get(mediator_var, np.nan)

            if np.isfinite(a) and np.isfinite(b):
                values.append(a * b)

        except Exception:
            continue

    values = np.asarray(values, dtype=float)

    if values.size == 0:
        return np.nan, np.nan, np.nan, 0

    ci_low, ci_high = np.quantile(values, [0.025, 0.975])

    return float(np.mean(values)), float(ci_low), float(ci_high), int(values.size)


# 整理样本
need = ["y", "treat", "city", "year"] + controls + [MEDIATOR]

d = (
    BASE_PANEL[need]
    .copy()
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

d = d[d["treat"].isin([0, 1])]

if d.empty:
    raise RuntimeError("中介效应检验样本为空，请检查变量缺失情况。")

rhs_c = " + ".join(controls) if controls else "1"

# 第一步：政策变量 -> 中介变量
formula_1 = f"{MEDIATOR} ~ treat + {rhs_c} + C(city) + C(year)"
model_1 = smf.ols(formula_1, data=d).fit(
    cov_type="cluster",
    cov_kwds={"groups": d["city"]},
)

# 第二步：政策变量 -> 被解释变量，总效应
formula_2 = f"y ~ treat + {rhs_c} + C(city) + C(year)"
model_2 = smf.ols(formula_2, data=d).fit(
    cov_type="cluster",
    cov_kwds={"groups": d["city"]},
)

# 第三步：政策变量 + 中介变量 -> 被解释变量
formula_3 = f"y ~ treat + {MEDIATOR} + {rhs_c} + C(city) + C(year)"
model_3 = smf.ols(formula_3, data=d).fit(
    cov_type="cluster",
    cov_kwds={"groups": d["city"]},
)

# 计算间接效应
a_hat = float(model_1.params.get("treat", np.nan))
b_hat = float(model_3.params.get(MEDIATOR, np.nan))
ab_hat = a_hat * b_hat if np.isfinite(a_hat) and np.isfinite(b_hat) else np.nan

ab_mean, ab_low, ab_high, boot_n = bootstrap_indirect_cluster(
    d,
    MEDIATOR,
    controls,
    B=BASE_B,
    seed=SEED,
)

total_effect = float(model_2.params.get("treat", np.nan))

if np.isfinite(ab_hat) and np.isfinite(total_effect) and total_effect != 0:
    mediated_share = ab_hat / total_effect
else:
    mediated_share = np.nan

support_mediation = (
    float(model_1.pvalues.get("treat", 1.0)) < 0.10
    and float(model_3.pvalues.get(MEDIATOR, 1.0)) < 0.10
    and np.isfinite(ab_low)
    and np.isfinite(ab_high)
    and ab_low * ab_high > 0
)

# 构造类似论文格式的表格
med_table = pd.DataFrame({
    "变量": [
        "因变量",
        "低碳城市试点政策变量",
        "",
        MEDIATOR_LABEL,
        "",
        "控制变量",
        "城市固定效应",
        "年份固定效应",
        "样本量",
        "R²",
        "Bootstrap间接效应",
        "间接效应占比",
        "是否支持中介效应",
    ],
    "(1)": [
        MEDIATOR_LABEL,
        fmt_coef(model_1, "treat"),
        "",
        "",
        "",
        "是",
        "是",
        "是",
        f"{int(model_1.nobs)}",
        f"{float(model_1.rsquared):.3f}",
        "",
        "",
        "",
    ],
    "(2)": [
        Y_LABEL,
        fmt_coef(model_2, "treat"),
        "",
        "",
        "",
        "是",
        "是",
        "是",
        f"{int(model_2.nobs)}",
        f"{float(model_2.rsquared):.3f}",
        "",
        "",
        "",
    ],
    "(3)": [
        Y_LABEL,
        fmt_coef(model_3, "treat"),
        "",
        fmt_coef(model_3, MEDIATOR),
        "",
        "是",
        "是",
        "是",
        f"{int(model_3.nobs)}",
        f"{float(model_3.rsquared):.3f}",
        f"{ab_hat:.4f} [{ab_low:.4f}, {ab_high:.4f}]" if np.isfinite(ab_hat) and np.isfinite(ab_low) and np.isfinite(ab_high) else "NA",
        f"{mediated_share * 100:.1f}%" if np.isfinite(mediated_share) else "NA",
        "是" if support_mediation else "否",
    ],
})

print("表X 机制分析：中介效应")
print("Table X Mechanism analysis: mediation effect")
print(f"Bootstrap重复抽样次数：{BASE_B}，有效抽样次数：{boot_n}")
print("注：括号内为按城市聚类的稳健标准误；****、***、**、* 分别表示在0.1%、1%、5%和10%的显著性水平上显著。")
print("注：Bootstrap置信区间基于按城市聚类重复抽样。")

display(
    med_table.style
    .set_properties(**{
        "font-family": "SimSun",
        "font-size": "12pt",
        "text-align": "center",
        "white-space": "pre-wrap",
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("font-family", "SimSun"),
                ("font-size", "12pt"),
                ("text-align", "center"),
            ],
        }
    ])
)

In [ ]:
# 机制分析：调节效应检验
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display

if "BASE_PANEL" not in globals() or "BEST_CONTROLS" not in globals():
    raise RuntimeError("请先运行第1段和第2段代码。")

controls = BEST_CONTROLS.copy()

moderators = [
    ("industry_upgrade_ratio", "产业结构高级化"),
    ("industry_rationalization_m2", "产业结构合理化"),
    ("share_secondary_pct", "第二产业占比"),
    ("share_tertiary_pct", "第三产业占比"),
]


def stars(p):
    """根据p值返回显著性星号。"""
    if p < 0.001:
        return "****"
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.10:
        return "*"
    return ""


def fmt_coef(coef, se, p):
    """输出系数和标准误。"""
    if not np.isfinite(coef):
        return ""

    return f"{coef:.4f}{stars(p)}\n({se:.4f})"


rows = []

for mvar, mlabel in moderators:
    need = ["y", "treat", "city", "year"] + controls + [mvar]

    d = (
        BASE_PANEL[need]
        .copy()
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    d = d[d["treat"].isin([0, 1])]

    if d.empty:
        continue

    # 构造交互项
    d["treat_m"] = d["treat"] * d[mvar]

    rhs = "treat + " + mvar + " + treat_m"

    if controls:
        rhs += " + " + " + ".join(controls)

    formula = f"y ~ {rhs} + C(city) + C(year)"

    model = smf.ols(
        formula,
        data=d,
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": d["city"]},
    )

    coef = float(model.params.get("treat_m", np.nan))
    se = float(model.bse.get("treat_m", np.nan))
    p = float(model.pvalues.get("treat_m", np.nan))

    if np.isfinite(coef) and coef > 0:
        direction = "正向调节"
    elif np.isfinite(coef) and coef < 0:
        direction = "负向调节"
    else:
        direction = "无法判断"

    rows.append({
        "调节变量": mlabel,
        "交互项Treat×M": fmt_coef(coef, se, p),
        "P值": f"{p:.4f}" if np.isfinite(p) else "",
        "调节方向": direction,
        "是否显著": "是" if np.isfinite(p) and p < 0.10 else "否",
        "样本量": int(model.nobs),
        "R²": f"{float(model.rsquared):.3f}",
    })

moderation_table = pd.DataFrame(rows)

print("表X 机制分析：调节效应")
print("Table X Mechanism analysis: moderation effect")
print("注：括号内为按城市聚类的稳健标准误；****、***、**、* 分别表示在0.1%、1%、5%和10%的显著性水平上显著。")
print("注：交互项 Treat×M 显著表示该变量对低碳城市试点政策效果具有调节作用。")

display(
    moderation_table.style
    .set_properties(**{
        "font-family": "SimSun",
        "font-size": "12pt",
        "text-align": "center",
        "white-space": "pre-wrap",
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("font-family", "SimSun"),
                ("font-size", "12pt"),
                ("text-align","center"),
            ],
        }
    ])
)

In [ ]:
# 可解释机器学习异质性分析：RF / XGBoost + SHAP
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings("ignore")

if "set_cn_plot_style" in globals():
    set_cn_plot_style()

try:
    from xgboost import XGBRegressor
except Exception as e:
    raise RuntimeError("请先安装 xgboost：pip install xgboost") from e

try:
    import shap
except Exception as e:
    raise RuntimeError("请先安装 shap：pip install shap") from e

if "BASE_PANEL" in globals():
    panel = BASE_PANEL.copy()
else:
    raw, _ = read_csv_auto("data.csv")
    panel = pd.DataFrame({
        "city": raw.iloc[:, 0].astype(str),
        "year": pd.to_numeric(raw.iloc[:, 2], errors="coerce"),
        "treat": pd.to_numeric(raw.iloc[:, 3], errors="coerce"),
        "y": pd.to_numeric(raw.iloc[:, 5], errors="coerce"),
        "ln_gdp_pc": pd.to_numeric(raw.iloc[:, 6], errors="coerce"),
        "ln_pop": pd.to_numeric(raw.iloc[:, 7], errors="coerce"),
        "urban": pd.to_numeric(raw.iloc[:, 8], errors="coerce"),
        "fdi_gdp": pd.to_numeric(raw.iloc[:, 9], errors="coerce"),
        "retail_gdp": pd.to_numeric(raw.iloc[:, 10], errors="coerce"),
    })

# 使用 treat + 基准控制变量
controls = []

if "BEST_CONTROLS" in globals() and isinstance(BEST_CONTROLS, (list, tuple)):
    controls = [c for c in BEST_CONTROLS if c in panel.columns]

if not controls:
    controls = ["ln_gdp_pc", "urban", "ln_pop", "fdi_gdp", "retail_gdp"]

features = ["treat"] + [c for c in controls if c != "treat"]

use_cols = ["y"] + features

d = panel[use_cols].copy()

for c in use_cols:
    d[c] = pd.to_numeric(d[c], errors="coerce")

d = d.replace([np.inf, -np.inf], np.nan).dropna()
d = d[d["treat"].isin([0, 1])]

final_features = [
    f for f in features
    if f in d.columns and d[f].nunique() > 1
]

X = d[final_features].copy()
y = d["y"].copy()

# 中文变量名
default_feature_labels = {
    "treat": "低碳城市试点政策变量",
    "ln_gdp_pc": "经济发展水平",
    "ln_pop": "城市规模",
    "urban": "城市化水平",
    "fdi_gdp": "对外开放水平",
    "retail_gdp": "市场消费水平",
}

feature_display_map = {}

for f in final_features:
    if "cn" in globals():
        feature_display_map[f] = cn(f)
    else:
        feature_display_map[f] = default_feature_labels.get(f, f)

print("机器学习样本量 =", len(d))
print("使用特征 =", [feature_display_map.get(v, v) for v in final_features])

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

# 参数选择
rf_base = RandomForestRegressor(
    random_state=42,
    n_jobs=1,
)

rf_param_grid = {
    "n_estimators": [300, 600],
    "max_depth": [3, 5, None],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["sqrt", 1.0],
}

rf_search = GridSearchCV(
    estimator=rf_base,
    param_grid=rf_param_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=1,
)

rf_search.fit(X_train, y_train)
rf_model = rf_search.best_estimator_

xgb_base = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=1,
)

xgb_param_grid = {
    "n_estimators": [300, 600],
    "learning_rate": [0.03, 0.05],
    "max_depth": [3, 4],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}

xgb_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=xgb_param_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=1,
)

xgb_search.fit(X_train, y_train)
xgb_model = xgb_search.best_estimator_

# 参数选择表
param_table = pd.DataFrame([
    {
        "模型": "随机森林",
        "最优参数": str(rf_search.best_params_),
        "交叉验证RMSE": f"{-rf_search.best_score_:.6f}",
    },
    {
        "模型": "XGBoost",
        "最优参数": str(xgb_search.best_params_),
        "交叉验证RMSE": f"{-xgb_search.best_score_:.6f}",
    },
])

print("表X 机器学习模型参数选择结果")

display(
    param_table.style
    .set_properties(**{
        "font-family": "SimSun",
        "font-size": "12pt",
        "text-align": "center",
        "white-space": "pre-wrap",
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("font-family", "SimSun"),
                ("font-size", "12pt"),
                ("text-align", "center"),
            ],
        }
    ])
)

# 回归指标表：R²、RMSE、MAE
def evaluate_model(model_name, model):
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    return {
        "模型": model_name,
        "训练集R²": f"{r2_score(y_train, y_pred_train):.6f}",
        "测试集R²": f"{r2_score(y_test, y_pred_test):.6f}",
        "测试集RMSE": f"{np.sqrt(mean_squared_error(y_test, y_pred_test)):.6f}",
        "测试集MAE": f"{mean_absolute_error(y_test, y_pred_test):.6f}",
    }

metrics_table = pd.DataFrame([
    evaluate_model("随机森林", rf_model),
    evaluate_model("XGBoost", xgb_model),
])

print("表X 机器学习模型回归指标")
print("注：本文选取R²、RMSE和MAE作为模型拟合效果评价指标。")

display(
    metrics_table.style
    .set_properties(**{
        "font-family": "SimSun",
        "font-size": "12pt",
        "text-align": "center",
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("font-family", "SimSun"),
                ("font-size", "12pt"),
                ("text-align", "center"),
            ],
        }
    ])
)

# 选择主模型：优先使用测试集R²更高的模型
rf_test_r2 = r2_score(y_test, rf_model.predict(X_test))
xgb_test_r2 = r2_score(y_test, xgb_model.predict(X_test))

if xgb_test_r2 >= rf_test_r2:
    main_model = xgb_model
    main_model_name = "XGBoost"
else:
    main_model = rf_model
    main_model_name = "随机森林"

print(f"SHAP解释主模型：{main_model_name}")

# SHAP样本
N_SHAP = min(2000, len(X_train))
X_shap = X_train.sample(N_SHAP, random_state=42) if len(X_train) > N_SHAP else X_train.copy()

X_shap_disp = X_shap.rename(columns=feature_display_map)

# 计算SHAP值
explainer = shap.TreeExplainer(main_model)

try:
    shap_values = explainer.shap_values(X_shap)
except Exception:
    shap_values = explainer(X_shap).values

if isinstance(shap_values, list):
    shap_values = shap_values[0]

# 图1：SHAP summary 图
plt.close("all")
plt.figure(figsize=(10, 6), dpi=600)

shap.summary_plot(
    shap_values,
    X_shap_disp,
    show=False,
    max_display=min(10, X_shap_disp.shape[1]),
)

plt.title(f"SHAP总结图（{main_model_name}）")
plt.tight_layout()
plt.show()
plt.close()

# 图2：SHAP总体交互图
N_INTER = min(800, len(X_shap))
X_inter = X_shap.sample(N_INTER, random_state=20260423) if len(X_shap) > N_INTER else X_shap.copy()
X_inter_disp = X_inter.rename(columns=feature_display_map)

try:
    inter_vals = explainer.shap_interaction_values(X_inter)

    if isinstance(inter_vals, list):
        inter_vals = inter_vals[0]

    plt.close("all")
    plt.figure(figsize=(12, 8), dpi=600)

    shap.summary_plot(
        inter_vals,
        X_inter_disp,
        max_display=min(7, X_inter_disp.shape[1]),
        show=False,
    )

    plt.suptitle(f"SHAP交互总结图（{main_model_name}）", y=1.02)
    plt.tight_layout()
    plt.show()
    plt.close()

except Exception as e:
    print("SHAP交互值计算失败，原因如下：")
    print(str(e))

# 图3：SHAP特征重要性图
mean_abs_shap = np.abs(shap_values).mean(axis=0)

importance_table = pd.DataFrame({
    "变量": [feature_display_map.get(c, c) for c in X_shap.columns],
    "平均绝对SHAP值": mean_abs_shap,
}).sort_values("平均绝对SHAP值", ascending=False)

print("表X SHAP特征重要性排序")

importance_table_show = importance_table.copy()
importance_table_show["平均绝对SHAP值"] = importance_table_show["平均绝对SHAP值"].map(lambda x: f"{x:.6f}")

display(
    importance_table_show.style
    .set_properties(**{
        "font-family": "SimSun",
        "font-size": "12pt",
        "text-align": "center",
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("font-family", "SimSun"),
                ("font-size", "12pt"),
                ("text-align", "center"),
            ],
        }
    ])
)

plt.close("all")
plt.figure(figsize=(8.5, 5.5), dpi=600)

plot_df = importance_table.sort_values("平均绝对SHAP值", ascending=True)

plt.barh(
    plot_df["变量"],
    plot_df["平均绝对SHAP值"],
)

plt.xlabel("平均绝对SHAP值")
plt.ylabel("变量")
plt.title(f"SHAP特征重要性图（{main_model_name}）")
plt.tight_layout()
plt.show()
plt.close()

# 保留关键对象，方便后续调用
ML_RF_MODEL = rf_model
ML_XGB_MODEL = xgb_model
ML_MAIN_MODEL = main_model
ML_MAIN_MODEL_NAME = main_model_name
ML_X_SHAP = X_shap
ML_SHAP_VALUES = shap_values
ML_EXPLAINER = explainer
ML_FEATURE_DISPLAY_MAP = feature_display_map